In [ ]:
"""
Minimal IMDb sentiment classification with a small instruction-tuned LM on CPU.

Edit PROMPT_TEMPLATE and VERBALIZER below to change behavior across runs.
Everything else stays fixed.
"""

import random
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

NUM_EXAMPLES = 25
SEED = 42
MODEL_NAME = "Qwen/Qwen3-0.6B"  # ~0.6B, smallest instruction-tuned model that can follow simple prompts

# Load random subset of data
print("Loading dataset...")
dataset = load_dataset("imdb", split="test")
random.seed(SEED)
indices = random.sample(range(len(dataset)), NUM_EXAMPLES)
samples = dataset.select(indices)
label_names = {0: "negative", 1: "positive"}

# Load model
print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, device_map="cpu"
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
VERBALIZER = {
    "positive": "positive",
    "Positive": "positive",
    "negative": "negative",
    "Negative": "negative",
    "Yes": "positive",
    "yes": "positive",
    "No": "negative",
    "no": "negative"
}

PROMPT_TEMPLATE = """\
Would you say the sentiment of this review is positive or negative? Just say "positive" or "negative".

Review: {text}"""

In [ ]:
correct = 0
total = 0
results = []

for i, example in enumerate(samples):
    text = example["text"][:1500]  # truncate long reviews to keep things fast
    gold = label_names[example["label"]]

    user_message = PROMPT_TEMPLATE.format(text=text)
    messages = [{"role": "user", "content": user_message}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            temperature=1.0,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1] :]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Map through verbalizer
    first_word = raw_output.split()[0].rstrip(".,!") if raw_output else ""
    pred = VERBALIZER.get(first_word, None)

    is_correct = pred == gold
    correct += int(is_correct)
    total += 1

    status = "✓" if is_correct else "✗" if pred else "?"
    print(f"[{i+1:3d}/{NUM_EXAMPLES}] {status}  gold={gold:<8s}  pred={str(pred):<8s}  raw='{raw_output}'")

    results.append({
        "index": indices[i],
        "gold": gold,
        "pred": pred,
        "raw_output": raw_output,
    })

# ── Summary ──────────────────────────────────────────────────────────────────

parsed = sum(1 for r in results if r["pred"] is not None)
print(f"\n{'='*60}")
print(f"Accuracy:    {correct}/{total} = {correct/total:.1%}")
print(f"Parse rate:  {parsed}/{total} = {parsed/total:.1%}")
print(f"Model:       {MODEL_NAME}")